# Graph Neural Networks (GNNs) - Workshop

Welcome to this comprehensive workshop on Graph Neural Networks (GNNs). This notebook will cover everything from the basic concepts of graphs to implementing and training state-of-the-art GNN models using PyTorch and PyTorch Geometric (PyG).

## 1. Introduction to Graphs

A **Graph** is a data structure consisting of **nodes** (or vertices) and **edges** (connections between nodes). They are perfect for modeling relations and processes in physical, biological, social, and information systems.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f'PyTorch version: {torch.__version__}')

## 2. Setting up our Environment

For modern GNNs, **PyTorch Geometric (PyG)** is the industry standard. It's built on top of PyTorch and provides highly optimized implementations of graph operations.

In [ ]:
# Run this cell to install PyTorch Geometric (uncomment if you haven't installed it)
# !pip install torch_geometric
# !pip install networkx matplotlib

import torch_geometric
import networkx as nx
import matplotlib.pyplot as plt

print(f'PyTorch Geometric version: {torch_geometric.__version__}')

## 3. Creating and Visualizing Graphs

Let's create a simple graph from scratch. In PyG, a graph is represented by a `Data` object, which takes:
- `x`: Node feature matrix
- `edge_index`: Graph connectivity (COO format)
- `y`: Target labels

In [ ]:
from torch_geometric.data import Data

# 1. Define edges (connections)
# edge_index represents [source_nodes, target_nodes]
edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.long)

# 2. Define node features (e.g., 3 nodes, each with 2 features)
x = torch.tensor([[-1., 0.], [0., 1.], [1., 0.]], dtype=torch.float)

# 3. Create the Data object
data = Data(x=x, edge_index=edge_index)
print(data)

Let's visualize this simple graph using NetworkX. PyG provides a convenient conversion function.

In [ ]:
from torch_geometric.utils import to_networkx

def visualize_graph(data):
    G = to_networkx(data, to_undirected=True)
    plt.figure(figsize=(6, 6))
    nx.draw(G, with_labels=True, node_color='skyblue', node_size=1500, font_weight='bold', font_size=15)
    plt.show()

visualize_graph(data)

## 4. The Cora Dataset: Node Classification

To see GNNs in action, we'll use the famous **Cora dataset**. It's a citation network where nodes are documents, and edges are citations. The goal is to classify the category of each document based on its text features and its citations.

In [ ]:
from torch_geometric.datasets import Planetoid

# Load the dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print(f'Dataset: {dataset.name}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Is undirected: {data.is_undirected()}')

## 5. Message Passing & Graph Convolutional Networks (GCNs)

The core of a GNN is the **Message Passing** paradigm. Nodes update their features by aggregating features from their neighbors. 

A Graph Convolutional Network (GCN) applies a simple weighted average of neighboring node features, followed by a linear transformation and a non-linearity.

In [ ]:
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        # First GCN Layer: Input features -> Hidden features
        self.conv1 = GCNConv(dataset.num_features, hidden_channels)
        # Second GCN Layer: Hidden features -> Classes
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        # 1. First message passing step
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # 2. Second message passing step
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)

## 6. Training the GNN

Training a GNN is very similar to training any PyTorch model. We'll use the cross-entropy loss function since this is a classification problem. Notice that we only calculate the loss on the `train_mask` elements.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    optimizer.zero_grad()  # Clear gradients
    out = model(data.x, data.edge_index)  # Perform a single forward pass
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes
    loss.backward()  # Derive gradients
    optimizer.step()  # Update parameters based on gradients
    return loss

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)  # Use the class with highest probability
    test_correct = pred[data.test_mask] == data.y[data.test_mask]  # Check against ground-truth labels
    test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  # Derive ratio of correct predictions
    return test_acc

# Train for 200 epochs
for epoch in range(1, 201):
    loss = train()
    if epoch % 10 == 0:
        acc = test()
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test Acc: {acc:.4f}')

## 7. Adding Graph Attention (GAT)

While GCNs assign fixed weights to neighbors based on the graph structure, Graph Attention Networks (GATs) learn which neighbors are more important during the message passing phase through attention mechanisms.

In [ ]:
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):
    def __init__(self, hidden_channels, heads):
        super(GAT, self).__init__()
        # First layer uses multiple attention heads
        self.conv1 = GATConv(dataset.num_features, hidden_channels, heads=heads, dropout=0.6)
        # Second layer outputs the class probabilities
        self.conv2 = GATConv(hidden_channels * heads, dataset.num_classes, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

gat_model = GAT(hidden_channels=8, heads=8)
optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=0.005, weight_decay=5e-4)

def train_gat():
    gat_model.train()
    optimizer_gat.zero_grad()
    out = gat_model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer_gat.step()
    return loss

def test_gat():
    gat_model.eval()
    out = gat_model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    test_correct = pred[data.test_mask] == data.y[data.test_mask]
    test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
    return test_acc

print('Training GAT Model...')
for epoch in range(1, 201):
    loss = train_gat()
    if epoch % 20 == 0:
        acc = test_gat()
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test Acc: {acc:.4f}')

## 8. Link Prediction (Optional/Advanced)

In Link Prediction, our goal is to predict missing edges in a graph. This is useful for recommendation systems (e.g., predicting that user A will like product B). This involves creating representations for pairs of nodes and using a classifier on those pairs.

In [ ]:
# For Link Prediction, typically we split the edges into train/val/test, 
# rather than splitting the nodes. This involves using transforms like `RandomLinkSplit`
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(is_undirected=True)
train_data, val_data, test_data = transform(data)

print(f'Train edges: {train_data.edge_label_index.size(1)}')
print(f'Test edges: {test_data.edge_label_index.size(1)}')
# (We leave the full implementation as an exercise for the workshop!)

## 9. Visualizing Performance

Let's evaluate the model beyond just accuracy by visualizing the graph predictions, confusion matrix, and t-SNE embeddings.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx

def visualize_prediction_graph(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct_mask = (pred == data.y).cpu().numpy()
    node_colors = ['#2ecc71' if is_correct else '#e74c3c' for is_correct in correct_mask]
    G = to_networkx(data.cpu(), to_undirected=True)
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=30, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='lightgray', alpha=0.3)
    plt.title("Cora Graph Predictions\n(Green: Correct ✅, Red: Incorrect ❌)", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()

visualize_prediction_graph(model, data)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_confusion_matrix(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    # We typically evaluate the test set (unseen nodes)
    y_pred = pred[data.test_mask].cpu().numpy()
    y_true = data.y[data.test_mask].cpu().numpy()
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(dataset.num_classes))
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap='Blues', ax=ax, colorbar=False)
    plt.title("Confusion Matrix on Test Set", fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Document Class')
    plt.ylabel('Actual Document Class')
    plt.show()

plot_confusion_matrix(model, data)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_embeds(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    # Use t-SNE to reduce the dimensions down to 2 components for 2D plotting
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(out.detach().cpu().numpy())
    ground_truth = data.y.cpu().numpy()
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                          c=ground_truth, cmap='Set1', s=30, alpha=0.8)
    plt.title("t-SNE Visualization of GNN Node Embeddings", fontsize=14, fontweight='bold')
    plt.legend(handles=scatter.legend_elements()[0], labels=[f"Class {i}" for i in range(dataset.num_classes)])
    plt.show()

visualize_embeds(model, data)

## Conclusion

In this workshop, we:
1. Learned what graphs are and how to represent them.
2. Built our own graphs in PyTorch Geometric.
3. Implemented a Graph Convolutional Network (GCN) for Node Classification.
4. Upgraded to a Graph Attention Network (GAT) to learn dynamic edge weights.
5. Explored how Link Prediction differs from Node Classification.

### Next Steps / Exercises
- Try applying GNNs to **Graph Classification** (predicting a property of an entire graph instead of individual nodes, e.g., in molecular property prediction).
- Explore other message passing architectures like **GraphSAGE** or **GIN**.